# 08 — Round-3 revision analyses (CPU only, no training)

Re-uses the saved per-image softmax outputs of all runs (notebook 02), the Grad-CAM per-image table
(notebook 05) and the lung boxes (notebook 07). Nothing is retrained. Typical runtime on a Kaggle CPU
session: 10–20 min (most of it is reading the 7,135 images once for the shortcut probes).

**Attach as inputs:** the results dataset(s) of notebook 02 (all `results/<run>/preds.csv` + `metrics.json`),
the output of notebook 05 (`gradcam/gradcam_per_image.csv`), the output of notebook 07
(`lungcrop/lung_boxes.csv`) and the original Kaggle dataset `jtiptj/chest-xray-pneumoniacovid19tuberculosis`.

| Output (in `/kaggle/working/round3/`) | Reviewer item |
|---|---|
| `pooled_attention.csv` — pooled hierarchical estimate (16 main / 8 robustness effects, and subgroups), 95% and 90% CIs, TOST at ±0.01 and ±0.02 | W1 |
| `equivalence_per_contrast.csv` — 90% CI and TOST for each of the 24 attention effects | W1 |
| `crossing_counts.csv` — replicates with Δ*≤0 and Δ*≥0 for every macro-F1 contrast | resolution of B |
| `mobilenet_matched_baseline.csv` — SE/CBAM vs the dropout-matched MobileNetV2 baseline | W4 |
| `collapsed_run_sensitivity.csv` — custom-CNN effects with/without seed 1, mean and median | collapsed run |
| `seed_medians.csv` — median over seeds for all 24 configurations | median summary |
| `shortcut_probes_ci.csv` — the four/six probes with stratified test-bootstrap CIs | probe CIs |
| `fig_gradcam_energy_v2.png` — per-class chance lines and n per box | Fig. 9 |
| `fig_cm_representative.png`, `fig_cm_all_{imbalanced,balanced}.png` | Fig. 6 / supplement |
| `per_seed_all_runs.csv` — macro F1 / accuracy of every run, both checkpoints | supplementary table |

Zip `/kaggle/working/round3` (last cell) and send the zip back.

In [ ]:
import os, glob, json, re, io, time, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

RESULT_GLOBS = ['/kaggle/input/**/results/*/metrics.json', '/kaggle/working/results/*/metrics.json']
OUT = '/kaggle/working/round3'; os.makedirs(OUT, exist_ok=True)
N_BOOT = 2000
CKPT = 'best'
RNG = np.random.default_rng(12345)                     # same seed as notebook 03
CLASS_NAMES = ['COVID19', 'NORMAL', 'PNEUMONIA', 'TURBERCULOSIS']; K = 4
SHORT = ['COV', 'NOR', 'PNE', 'TB']
ARCH_ORDER = ['resnet50', 'densenet121', 'mobilenetv2', 'customcnn']
ARCH_LABEL = {'resnet50': 'ResNet50', 'densenet121': 'DenseNet121', 'mobilenetv2': 'MobileNetV2', 'customcnn': 'Custom CNN'}
ATT_LABEL = {'none': 'Baseline', 'se': 'SE', 'cbam': 'CBAM'}
MARGINS = [0.01, 0.02]

In [ ]:
# ── load runs (identical to notebook 03) ─────────────────────────────────────
metas = {}
for g in RESULT_GLOBS:
    for f in glob.glob(g, recursive=True):
        m = json.load(open(f))
        if m.get('status') != 'done' or (m.get('smoke_test') and os.environ.get('ALLOW_SMOKE') != '1'): continue
        m['dir'] = os.path.dirname(f); metas.setdefault(m['run_id'], m)
print(len(metas), 'finished runs')
RUNS, ref_paths = {}, None
for rid, m in sorted(metas.items()):
    P = pd.read_csv(os.path.join(m['dir'], 'preds.csv'))
    rec = dict(meta=m, variant=''.join(f'_{k}-{v}' for k, v in sorted(m.get('opts', {}).items())))
    for ck in ['best', 'final']:
        d = P[(P.split == 'test') & (P.ckpt == ck)].sort_values('idx')
        rec[(ck, 'prob')] = d[[f'p_{c}' for c in CLASS_NAMES]].values
        rec['y'] = d.y_true.values
        if ck == 'best':
            if ref_paths is None: ref_paths = d.relpath.tolist()
            assert d.relpath.tolist() == ref_paths, f'{rid}: test order differs'
    RUNS[rid] = rec
Y = RUNS[next(iter(RUNS))]['y']; NTEST = len(Y)
cfg_of = lambda r: (r['meta']['arch'], r['meta']['attention'], r['meta']['condition'], r['variant'])
CFG = {}
for rid, r in RUNS.items(): CFG.setdefault(cfg_of(r), {})[r['meta']['seed']] = rid

def metrics_batch(cm):
    cm = cm.astype(float); tp = np.diagonal(cm, axis1=-2, axis2=-1)
    fp = cm.sum(-2) - tp; fn = cm.sum(-1) - tp
    with np.errstate(divide='ignore', invalid='ignore'):
        prec = np.nan_to_num(tp / (tp + fp)); rec = np.nan_to_num(tp / (tp + fn)); f1 = np.nan_to_num(2 * prec * rec / (prec + rec))
    return {'acc': tp.sum(-1) / cm.sum((-1, -2)), 'macro_f1': f1.mean(-1)}

def cm_of(y, p, idx=None):
    if idx is None: return np.bincount(y * K + p, minlength=K * K).reshape(K, K)
    code = y[idx] * K + p[idx]; off = (np.arange(idx.shape[0]) * K * K)[:, None]
    return np.bincount((code + off).ravel(), minlength=idx.shape[0] * K * K).reshape(-1, K, K)

strata = [np.where(Y == c)[0] for c in range(K)]
BOOT = np.concatenate([s[RNG.integers(0, len(s), (N_BOOT, len(s)))] for s in strata], axis=1)   # same draw as notebook 03
POINT, BOOTM = {}, {}
for rid, r in RUNS.items():
    for ck in ['best', 'final']:
        p = r[(ck, 'prob')].argmax(1)
        POINT[(rid, ck)] = metrics_batch(cm_of(Y, p)); BOOTM[(rid, ck)] = metrics_batch(cm_of(Y, p, BOOT))
seeds_of = lambda cfg: sorted(CFG.get(cfg, {}))
print('bootstrap ready')

In [ ]:
# ── per-seed table (supplement) ──────────────────────────────────────────────
rows = []
for rid, r in sorted(RUNS.items()):
    m = r['meta']
    rows.append(dict(run_id=rid, arch=m['arch'], attention=m['attention'], condition=m['condition'], variant=r['variant'],
                     seed=m['seed'], f1_best=POINT[(rid, 'best')]['macro_f1'], f1_final=POINT[(rid, 'final')]['macro_f1'],
                     acc_best=POINT[(rid, 'best')]['acc'], acc_final=POINT[(rid, 'final')]['acc']))
pd.DataFrame(rows).to_csv(f'{OUT}/per_seed_all_runs.csv', index=False)

In [ ]:
# ── generic paired hierarchical contrast with an optional seed subset ─────────
def contrast_dist(pairs, seeds, ck=CKPT, metric='macro_f1', pick=None):
    """pairs: list of (cfg_a, cfg_b, weight). Returns (point, (N_BOOT,) dist) of Σ w·mean_s[a_s − b_s].
    One image resample (BOOT) and ONE seed resample per replicate, shared by all pairs (keeps their correlation)."""
    S = len(seeds)
    if pick is None: pick = RNG.integers(0, S, (N_BOOT, S))
    tot_b = np.zeros(N_BOOT); tot_p = 0.0
    for a, b, w in pairs:
        Da = np.stack([BOOTM[(CFG[a][s], ck)][metric] for s in seeds]); Db = np.stack([BOOTM[(CFG[b][s], ck)][metric] for s in seeds])
        tot_b += w * (Da - Db)[pick, np.arange(N_BOOT)[:, None]].mean(1)
        tot_p += w * np.mean([POINT[(CFG[a][s], ck)][metric] - POINT[(CFG[b][s], ck)][metric] for s in seeds])
    return tot_p, tot_b

def summarise(label, point, d, extra=None):
    r = dict(effect=label, delta=point, ci95_lo=np.percentile(d, 2.5), ci95_hi=np.percentile(d, 97.5),
             ci90_lo=np.percentile(d, 5), ci90_hi=np.percentile(d, 95),
             n_le0=int((d <= 0).sum()), n_ge0=int((d >= 0).sum()), B=len(d),
             p_boot=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
    for m in MARGINS:     # TOST at level 0.05 = 90% CI inside (−m, m)
        r[f'equiv_{m}'] = bool(r['ci90_lo'] > -m and r['ci90_hi'] < m)
        r[f'p_tost_{m}'] = max(((d <= -m).mean()), ((d >= m).mean()))
    if extra: r.update(extra)
    return r

MAIN = [((a, t, c, ''), (a, 'none', c, ''), f'{ATT_LABEL[t]} | {ARCH_LABEL[a]} | {c}', a, t, c)
        for c in ['imbalanced', 'balanced'] for a in ARCH_ORDER for t in ['se', 'cbam']
        if (a, t, c, '') in CFG and (a, 'none', c, '') in CFG]
ROB_VARIANTS = [('resnet50', '_lungcrop-True', 'lung crop'), ('resnet50', '_val-grouped', 'grouped validation'),
                ('resnet50', '_placement-multi', 'multi-depth'), ('densenet121', '_pretrain-xrv', 'CXR pretraining')]
ROB = []
for a, v, lab in ROB_VARIANTS:
    for t in ['se', 'cbam']:
        base = (a, 'none', 'imbalanced', v if v != '_placement-multi' else '')   # multi-depth vs single-position baseline
        att = (a, t, 'imbalanced', v)
        if att in CFG and base in CFG: ROB.append((att, base, f'{ATT_LABEL[t]} | {ARCH_LABEL[a]} | {lab}', a, t, lab))
print(len(MAIN), 'main and', len(ROB), 'robustness attention effects')

In [ ]:
# ── W1: per-contrast 90% CIs / TOST, and pooled hierarchical estimates ───────
per, pooled = [], []
for att, base, lab, a, t, c in MAIN + ROB:
    seeds = sorted(set(seeds_of(att)) & set(seeds_of(base)))
    pt, d = contrast_dist([(att, base, 1.0)], seeds)
    per.append(summarise(lab, pt, d, dict(family='main' if (att, base, lab, a, t, c) in MAIN else 'robustness', n_seeds=len(seeds))))
EQ = pd.DataFrame(per); EQ.to_csv(f'{OUT}/equivalence_per_contrast.csv', index=False)

def pool(label, items):
    if not items: return
    seeds = sorted(set.intersection(*[set(seeds_of(x[0])) & set(seeds_of(x[1])) for x in items]))
    pt, d = contrast_dist([(x[0], x[1], 1.0 / len(items)) for x in items], seeds)
    pooled.append(summarise(label, pt, d, dict(n_effects=len(items), n_seeds=len(seeds))))
pool(f'all {len(MAIN)} main attention effects', MAIN)
pool('main, SE (8)', [x for x in MAIN if x[4] == 'se']); pool('main, CBAM (8)', [x for x in MAIN if x[4] == 'cbam'])
pool('main, imbalanced (8)', [x for x in MAIN if x[5] == 'imbalanced']); pool('main, balanced (8)', [x for x in MAIN if x[5] == 'balanced'])
pool('main, pretrained backbones (12)', [x for x in MAIN if x[3] != 'customcnn'])
for a in ARCH_ORDER: pool(f'main, {ARCH_LABEL[a]} (4)', [x for x in MAIN if x[3] == a])
if ROB: pool(f'all {len(ROB)} robustness attention effects', ROB)
# sensitivity: pooled main estimate without the collapsed custom-CNN run (drop that seed for the custom CNN only)
POOL = pd.DataFrame(pooled); POOL.to_csv(f'{OUT}/pooled_attention.csv', index=False)
pd.set_option('display.width', 250); print(POOL.round(4).to_string()); print(EQ.round(4).to_string())

In [ ]:
# ── replicate crossing counts for every macro-F1 contrast family used in the paper ──
# (the ATTENTION rows come from EQ; balancing / SE−CBAM / interaction / ablation recomputed here)
cross = [dict(family='ATTENTION', effect=r.effect, delta=r.delta, ci95_lo=r.ci95_lo, ci95_hi=r.ci95_hi, n_le0=r.n_le0, n_ge0=r.n_ge0, B=r.B)
         for r in EQ.itertuples()]
for c in ['imbalanced', 'balanced']:
    for a in ARCH_ORDER:
        if (a, 'se', c, '') not in CFG or (a, 'cbam', c, '') not in CFG: continue
        pt, d = contrast_dist([((a, 'se', c, ''), (a, 'cbam', c, ''), 1.0)], seeds_of((a, 'se', c, '')))
        s = summarise(f'SE−CBAM | {ARCH_LABEL[a]} | {c}', pt, d); s['family'] = 'SE-vs-CBAM'; cross.append(s)
for a in ARCH_ORDER:
    for t in ['none', 'se', 'cbam']:
        if (a, t, 'balanced', '') not in CFG or (a, t, 'imbalanced', '') not in CFG: continue
        pt, d = contrast_dist([((a, t, 'balanced', ''), (a, t, 'imbalanced', ''), 1.0)], seeds_of((a, t, 'balanced', '')))
        s = summarise(f'balancing | {ARCH_LABEL[a]} {ATT_LABEL[t]}', pt, d); s['family'] = 'BALANCING'; cross.append(s)
CROSS = pd.DataFrame(cross)[['family', 'effect', 'delta', 'ci95_lo', 'ci95_hi', 'n_le0', 'n_ge0', 'B']]
CROSS.to_csv(f'{OUT}/crossing_counts.csv', index=False)

In [ ]:
# ── W4: MobileNetV2 attention effects against the dropout-matched baseline ───
mb = []
matched = ('mobilenetv2', 'none', 'imbalanced', '_head_dropout-0.0')
if matched in CFG:
    for t in ['se', 'cbam']:
        att = ('mobilenetv2', t, 'imbalanced', ''); seeds = sorted(set(seeds_of(att)) & set(seeds_of(matched)))
        pt, d = contrast_dist([(att, matched, 1.0)], seeds); mb.append(summarise(f'{ATT_LABEL[t]} − matched (no-dropout) baseline | imbalanced', pt, d, dict(n_seeds=len(seeds))))
        pt, d = contrast_dist([(att, ('mobilenetv2', 'none', 'imbalanced', ''), 1.0)], seeds)
        mb.append(summarise(f'{ATT_LABEL[t]} − dropout baseline, same seeds | imbalanced', pt, d, dict(n_seeds=len(seeds))))
pd.DataFrame(mb).to_csv(f'{OUT}/mobilenet_matched_baseline.csv', index=False); print(pd.DataFrame(mb).round(4).to_string())

In [ ]:
# ── collapsed custom-CNN + SE run: with / without the collapsed seed; medians ─
coll = [(rid, POINT[(rid, 'best')]['macro_f1']) for rid in CFG.get(('customcnn', 'se', 'imbalanced', ''), {}).values()]
HAVE_CC = bool(coll) and all(c in CFG for c in [('customcnn', t, 'imbalanced', '') for t in ['none', 'se', 'cbam']])
if not HAVE_CC: print('custom-CNN imbalanced configurations missing — skipping collapsed-run block')
if HAVE_CC:
    crid = min(coll, key=lambda x: x[1])[0]; cseed = RUNS[crid]['meta']['seed']; print('collapsed run:', crid)
    cs = []
    allseeds = seeds_of(('customcnn', 'se', 'imbalanced', '')); keep = [s for s in allseeds if s != cseed]
    for lab, a, b in [('SE − Baseline', ('customcnn', 'se', 'imbalanced', ''), ('customcnn', 'none', 'imbalanced', '')),
                      ('SE − CBAM', ('customcnn', 'se', 'imbalanced', ''), ('customcnn', 'cbam', 'imbalanced', ''))]:
        for name, seeds in [('all 5 seeds', allseeds), (f'without seed {cseed}', keep)]:
            pt, d = contrast_dist([(a, b, 1.0)], seeds)
            diffs = [POINT[(CFG[a][s], 'best')]['macro_f1'] - POINT[(CFG[b][s], 'best')]['macro_f1'] for s in seeds]
            cs.append(summarise(f'Custom CNN imbalanced | {lab} | {name}', pt, d, dict(median_of_seed_differences=float(np.median(diffs)), n_seeds=len(seeds))))
    v = [POINT[(CFG[('customcnn', 'se', 'imbalanced', '')][s], 'best')]['macro_f1'] for s in keep]
    cs.append(dict(effect=f'Custom CNN + SE imbalanced, macro F1 without seed {cseed}', delta=np.mean(v), sd=np.std(v, ddof=1), median=np.median(v)))
    pd.DataFrame(cs).to_csv(f'{OUT}/collapsed_run_sensitivity.csv', index=False); print(pd.DataFrame(cs).round(4).to_string())
    # pooled main estimate without the collapsed seed for the custom CNN
    items_wo = [x for x in MAIN if not (x[3] == 'customcnn' and x[5] == 'imbalanced')]
    pt_all, d_all = contrast_dist([(x[0], x[1], 1.0) for x in items_wo], allseeds)
    pt_cc, d_cc = contrast_dist([(x[0], x[1], 1.0) for x in MAIN if x[3] == 'customcnn' and x[5] == 'imbalanced'], keep)
    d_tot = (d_all + d_cc) / len(MAIN); p_tot = (pt_all + pt_cc) / len(MAIN)
    POOL2 = pd.concat([POOL, pd.DataFrame([summarise(f'all {len(MAIN)} main effects, custom-CNN imbalanced without seed {cseed}', p_tot, d_tot, dict(n_effects=len(MAIN)))])])
    POOL2.to_csv(f'{OUT}/pooled_attention.csv', index=False)

med = []
for (a, t, c, v), d in CFG.items():
    if v: continue
    x = [POINT[(rid, 'best')]['macro_f1'] for rid in d.values()]
    med.append(dict(arch=ARCH_LABEL[a], attention=ATT_LABEL[t], condition=c, mean=np.mean(x), median=np.median(x), sd=np.std(x, ddof=1), n=len(x)))
pd.DataFrame(med).sort_values(['condition', 'arch', 'attention']).to_csv(f'{OUT}/seed_medians.csv', index=False)

In [ ]:
# ── confusion matrices: four representative panels + full grids ─────────────
def mean_cm(cfg, ck=CKPT):
    return np.mean([cm_of(Y, RUNS[CFG[cfg][s]][(ck, 'prob')].argmax(1)) for s in seeds_of(cfg)], 0)
def draw(ax, cm, title):
    ax.imshow(cm / cm.sum(1, keepdims=True), cmap='Blues', vmin=0, vmax=1)
    for i in range(K):
        for j in range(K):
            ax.text(j, i, f'{cm[i, j]:.1f}', ha='center', va='center', fontsize=8, color='white' if cm[i, j] / cm[i].sum() > .5 else 'black')
    ax.set_xticks(range(K)); ax.set_yticks(range(K)); ax.set_xticklabels(SHORT, fontsize=8); ax.set_yticklabels(SHORT, fontsize=8)
    ax.set_title(title, fontsize=9); ax.set_xlabel('predicted', fontsize=8); ax.set_ylabel('true', fontsize=8)
REP_ALL = [(('resnet50', 'none', 'imbalanced', ''), '(a) ResNet50 baseline, imbalanced'),
       (('resnet50', 'cbam', 'balanced', ''), '(b) ResNet50 + CBAM, balanced'),
       (('mobilenetv2', 'none', 'imbalanced', ''), '(c) MobileNetV2 baseline, imbalanced'),
       (('customcnn', 'se', 'imbalanced', ''), '(d) Custom CNN + SE, imbalanced')]
REP = [x for x in REP_ALL if x[0] in CFG]
fig, ax = plt.subplots(1, max(1, len(REP)), figsize=(13, 3.4), squeeze=False); ax = ax[0]
for k, (cfg, ttl) in enumerate(REP): draw(ax[k], mean_cm(cfg), ttl)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_cm_representative.png', dpi=300); plt.close()
for c in ['imbalanced', 'balanced']:
    fig, ax = plt.subplots(4, 3, figsize=(10, 13))
    for i, a in enumerate(ARCH_ORDER):
        for j, t in enumerate(['none', 'se', 'cbam']):
            if (a, t, c, '') in CFG: draw(ax[i, j], mean_cm((a, t, c, '')), f'{ARCH_LABEL[a]} {ATT_LABEL[t]}')
            else: ax[i, j].axis('off')
    plt.tight_layout(); plt.savefig(f'{OUT}/fig_cm_all_{c}.png', dpi=200); plt.close()

In [ ]:
# ── Grad-CAM box plot with per-class chance lines and n per box ──────────────
gf = glob.glob('/kaggle/input/**/gradcam_per_image.csv', recursive=True) + glob.glob('/kaggle/working/**/gradcam_per_image.csv', recursive=True)
if gf:
    G = pd.read_csv(gf[0]); g = G[G.qc_ok]
    names = list(G.model.unique()); cols = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
    chance = g.drop_duplicates('img').groupby('true').lung_area_frac.mean()
    nbox = g.groupby(['model', 'true']).size()
    fig, ax = plt.subplots(figsize=(max(9, 1.9 * len(names)), 4.8)); ticks = []
    for k, name in enumerate(names):
        for ci, c in enumerate(CLASS_NAMES):
            x = k * 4.6 + ci; dat = g[(g.model == name) & (g.true == c)].energy_in_lung.values
            b = ax.boxplot([dat], positions=[x], widths=.7, showfliers=False, patch_artist=True)
            b['boxes'][0].set(facecolor=cols[ci], alpha=.55)
            ax.plot([x - .4, x + .4], [chance[c]] * 2, ls=':', c='k', lw=1.2)
            ax.text(x, 0.015, f'{nbox.get((name, c), 0)}', ha='center', fontsize=6, rotation=90)
        ticks.append(k * 4.6 + 1.5)
    ax.set_xticks(ticks); ax.set_xticklabels(names, rotation=15, fontsize=8); ax.set_ylim(0, 1)
    ax.set_ylabel('fraction of Grad-CAM energy inside lungs')
    from matplotlib.patches import Patch
    ax.legend([Patch(color=c, alpha=.55) for c in cols] + [plt.Line2D([], [], ls=':', c='k')], SHORT + ['class lung-area fraction (chance)'],
              fontsize=7, ncol=5, loc='upper right')
    plt.tight_layout(); plt.savefig(f'{OUT}/fig_gradcam_energy_v2.png', dpi=300); plt.close()
    chance.rename('mean_lung_area_fraction').to_frame().assign(n_images=g.drop_duplicates('img').groupby('true').size()).to_csv(f'{OUT}/gradcam_class_chance.csv')
    nbox.rename('n_image_seed_pairs').to_csv(f'{OUT}/gradcam_box_n.csv')
else:
    print('gradcam_per_image.csv not found — attach the notebook-05 output to redraw Fig. 9')

In [ ]:
# ── shortcut probes with stratified test-bootstrap CIs (fixed fitted probes) ──
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, accuracy_score
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
cand = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/test/TURBERCULOSIS', recursive=True)})
cand = [c for c in cand if 'balanced' not in c.lower()] or cand
ROOT = cand[0]; print('dataset root:', ROOT)
rows = []
for sp in ['train', 'val', 'test']:
    for c in CLASS_NAMES:
        d = os.path.join(ROOT, sp, c)
        if not os.path.isdir(d): continue
        for fn in sorted(os.listdir(d)):
            if fn.lower().endswith(IMG_EXTS): rows.append(dict(split=sp, cls=c, path=os.path.join(d, fn), relpath=os.path.relpath(os.path.join(d, fn), ROOT), file=fn))
D = pd.DataFrame(rows); print(len(D), 'images')
T0 = time.time(); meta_rows, thumbs = [], []
for p in D.path:
    raw = open(p, 'rb').read(); im = Image.open(io.BytesIO(raw)); w, h = im.size
    meta_rows.append(dict(width=w, height=h, aspect=w / h, bytes=len(raw), ext=os.path.splitext(p)[1].lower(), mode=im.mode))
    thumbs.append(np.asarray(im.convert('RGB').convert('L').resize((32, 32), Image.BILINEAR), np.float32) / 255.)
D = pd.concat([D, pd.DataFrame(meta_rows)], axis=1); TH = np.stack(thumbs); print(f'read in {(time.time() - T0) / 60:.1f} min')
y = D.cls.map({c: i for i, c in enumerate(CLASS_NAMES)}).values; tr, te = (D.split == 'train').values, (D.split == 'test').values
yte = y[te]; st = [np.where(yte == c)[0] for c in range(K)]
PB = np.concatenate([s[np.random.default_rng(7).integers(0, len(s), (N_BOOT, len(s)))] for s in st], axis=1)
def ci(pred):
    f = metrics_batch(cm_of(yte, pred, PB))['macro_f1']; a = metrics_batch(cm_of(yte, pred, PB))['acc']
    return dict(test_acc=accuracy_score(yte, pred), test_macro_f1=f1_score(yte, pred, average='macro'),
                f1_ci_lo=np.percentile(f, 2.5), f1_ci_hi=np.percentile(f, 97.5), acc_ci_lo=np.percentile(a, 2.5), acc_ci_hi=np.percentile(a, 97.5))
def border(t, keep=0.2):
    t = t.copy(); b = int(t.shape[-1] * keep); t[..., b:-b, b:-b] = 0; return t
def lr_probe(X):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5, class_weight='balanced'))
    clf.fit(X[tr].reshape(tr.sum(), -1), y[tr]); return clf.predict(X[te].reshape(te.sum(), -1))
res = []
M_all = pd.get_dummies(D[['width', 'height', 'aspect', 'bytes', 'ext', 'mode']], columns=['ext', 'mode']).astype(float).values
pm = RandomForestClassifier(500, random_state=0, n_jobs=-1, class_weight='balanced').fit(M_all[tr], y[tr]).predict(M_all[te])
res.append(dict(probe='file metadata (width, height, aspect, bytes, extension, colour mode)', pixel_accessible='no (file-level)', **ci(pm)))
M_dim = D[['width', 'height', 'aspect']].values.astype(float)
pd_ = RandomForestClassifier(500, random_state=0, n_jobs=-1, class_weight='balanced').fit(M_dim[tr], y[tr]).predict(M_dim[te])
res.append(dict(probe='native resolution only (width, height, aspect)', pixel_accessible='partly (lost by the 256×256 resize except as interpolation/aspect distortion)', **ci(pd_)))
res.append(dict(probe='32×32 thumbnail, border only — full image', pixel_accessible='yes', **ci(lr_probe(border(TH)))))
res.append(dict(probe='32×32 thumbnail — full image', pixel_accessible='yes', **ci(lr_probe(TH))))
bf = glob.glob('/kaggle/input/**/lungcrop/lung_boxes.csv', recursive=True) + glob.glob('/kaggle/working/lungcrop/lung_boxes.csv')
if bf:
    Bx = pd.read_csv(bf[0]).set_index('relpath').reindex(D.relpath)
    CR = []
    for p, r in zip(D.path, Bx.itertuples()):
        im = Image.open(p).convert('RGB').convert('L'); W, H = im.size
        CR.append(np.asarray(im.crop((int(r.x0 * W), int(r.y0 * H), int(r.x1 * W), int(r.y1 * H))).resize((32, 32), Image.BILINEAR), np.float32) / 255.)
    CR = np.stack(CR)
    res.append(dict(probe='32×32 thumbnail — lung crop', pixel_accessible='yes', **ci(lr_probe(CR))))
    res.append(dict(probe='32×32 thumbnail, border only — lung crop', pixel_accessible='yes', **ci(lr_probe(border(CR)))))
PR = pd.DataFrame(res); PR.to_csv(f'{OUT}/shortcut_probes_ci.csv', index=False); print(PR.round(4).to_string())

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/round3', 'zip', OUT); print('wrote /kaggle/working/round3.zip — send this file back')